In [1]:
#Cell 1 — Imports

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 260
NUM_CLASSES = 3
CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]
SEED = 42
N_PER_CLASS = 15          # 15 x 3 classes = 45 test images evaluated
STEPS = 20                # 5% increments for deletion/insertion

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATASET_DIR = PROJECT_DIR / "dataset" / "processed_dataset"
TEST_DIR = DATASET_DIR / "test"

PNEUMOXNET_PATH = PROJECT_DIR / "models" / "pneumoxnet_seed42_best_acc.pth"
BASELINE_PATH = PROJECT_DIR / "models" / "efficientnetb2_seed42_best.pth"

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
XAI_METRICS_CSV = RESULTS_DIR / "xai_deletion_insertion_metrics.csv"

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

print("Device:", DEVICE)
print("PneumoXNet checkpoint:", PNEUMOXNET_PATH, "| exists:", PNEUMOXNET_PATH.exists())
print("Baseline checkpoint  :", BASELINE_PATH, "| exists:", BASELINE_PATH.exists())

Device: cuda
PneumoXNet checkpoint: /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/pneumoxnet_seed42_best_acc.pth | exists: True
Baseline checkpoint  : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/efficientnetb2_seed42_best.pth | exists: True


In [3]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
assert test_dataset.classes == CLASS_NAMES, f"Class order mismatch: {test_dataset.classes}"
print("Test samples:", len(test_dataset))

Test samples: 879


In [4]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention()
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

class MultiScaleFeatureFusion(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        mid = in_channels // reduction
        self.reduce = nn.Sequential(nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_1x1 = nn.Sequential(nn.Conv2d(mid, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_3x3 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_5x5 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.fusion = nn.Sequential(nn.Conv2d(mid * 3, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True))
    def forward(self, x):
        x = self.reduce(x)
        f = torch.cat([self.branch_1x1(x), self.branch_3x3(x), self.branch_5x5(x)], dim=1)
        return self.fusion(f)

class AdaptiveFeatureFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight_generator = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid()
        )
    def forward(self, feature_a, feature_b):
        weights = self.weight_generator(torch.cat([feature_a, feature_b], dim=1))
        return weights * feature_a + (1.0 - weights) * feature_b

class ResidualEnhancement(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
    def forward(self, x):
        return x + self.refine(x)

class PneumoXNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        weights = EfficientNet_B2_Weights.DEFAULT
        backbone = efficientnet_b2(weights=weights)
        self.backbone = backbone.features
        self.feature_channels = 1408
        self.cbam = CBAM(self.feature_channels)
        self.multiscale = MultiScaleFeatureFusion(self.feature_channels)
        self.aff = AdaptiveFeatureFusion(self.feature_channels)
        self.residual = ResidualEnhancement(self.feature_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.55),
            nn.Linear(self.feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.45),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        features = self.backbone(x)
        cbam_out = self.cbam(features)
        ms_out = self.multiscale(features)
        fused = self.aff(cbam_out, ms_out)
        enhanced = self.residual(fused)
        pooled = self.pool(enhanced)
        return self.classifier(pooled)

In [5]:
pneumoxnet_model = PneumoXNet(num_classes=NUM_CLASSES).to(DEVICE)
pneumoxnet_model.load_state_dict(torch.load(PNEUMOXNET_PATH, map_location=DEVICE))
pneumoxnet_model.eval()

baseline_model = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)
in_features = baseline_model.classifier[1].in_features
baseline_model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, NUM_CLASSES))
baseline_model.load_state_dict(torch.load(BASELINE_PATH, map_location=DEVICE))
baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

print("Both models loaded successfully.")

Both models loaded successfully.


In [6]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_index):
        self.model.zero_grad()
        outputs = self.model(input_tensor)
        outputs[:, class_index].backward()
        weights = self.gradients[0].mean(dim=(1, 2))
        cam = torch.zeros(self.activations.shape[2:], device=input_tensor.device)
        for i, w in enumerate(weights):
            cam += w * self.activations[0, i]
        cam = torch.relu(cam)
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        return cam.cpu().numpy()

pneumoxnet_gradcam = GradCAM(pneumoxnet_model, pneumoxnet_model.residual)
baseline_gradcam = GradCAM(baseline_model, baseline_model.features[-1])

In [10]:
# ============================================================
# Cell 7: Deletion/Insertion Metric Functions
# ============================================================

def denormalize(tensor):
    return tensor.cpu() * IMAGENET_STD + IMAGENET_MEAN

def normalize(tensor):
    return (tensor - IMAGENET_MEAN) / IMAGENET_STD

def get_blurred_baseline(image_tensor):
    img = denormalize(image_tensor).clamp(0, 1).permute(1, 2, 0).numpy()
    blurred = cv2.GaussianBlur(img, (51, 51), 0)
    blurred_tensor = torch.from_numpy(blurred).permute(2, 0, 1).float()
    return normalize(blurred_tensor)

@torch.no_grad()
def predict_prob(model, input_tensor, class_index):
    outputs = model(input_tensor)
    probs = torch.softmax(outputs, dim=1)
    return probs[0, class_index].item()

def deletion_insertion_auc(model, gradcam, image_tensor, device, steps=STEPS):
    input_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        pred_class = model(input_tensor).argmax(dim=1).item()

    cam = gradcam.generate(input_tensor, pred_class)
    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))

    order = np.argsort(-cam_resized.flatten())  # most important first
    total_pixels = IMAGE_SIZE * IMAGE_SIZE
    step_size = total_pixels // steps

    baseline_tensor = get_blurred_baseline(image_tensor)

    original_flat = image_tensor.clone().view(3, -1)
    baseline_flat = baseline_tensor.clone().view(3, -1)

    deletion_probs, insertion_probs = [], []

    for step in range(steps + 1):
        n_pixels = min(step * step_size, total_pixels)
        idx = order[:n_pixels]

        del_img = original_flat.clone()
        del_img[:, idx] = baseline_flat[:, idx]
        del_tensor = del_img.view(3, IMAGE_SIZE, IMAGE_SIZE).unsqueeze(0).to(device)
        deletion_probs.append(predict_prob(model, del_tensor, pred_class))

        ins_img = baseline_flat.clone()
        ins_img[:, idx] = original_flat[:, idx]
        ins_tensor = ins_img.view(3, IMAGE_SIZE, IMAGE_SIZE).unsqueeze(0).to(device)
        insertion_probs.append(predict_prob(model, ins_tensor, pred_class))

    x = np.linspace(0, 1, steps + 1)
    trapz_fn = getattr(np, "trapezoid", None) or np.trapz
    deletion_auc = trapz_fn(deletion_probs, x)
    insertion_auc = trapz_fn(insertion_probs, x)

    return deletion_auc, insertion_auc, pred_class

In [11]:
indices_by_class = {i: [] for i in range(NUM_CLASSES)}
for idx, (_, label) in enumerate(test_dataset.samples):
    indices_by_class[label].append(idx)

rng = np.random.default_rng(SEED)
selected_indices = []
for class_idx, idxs in indices_by_class.items():
    chosen = rng.choice(idxs, size=min(N_PER_CLASS, len(idxs)), replace=False)
    selected_indices.extend(chosen.tolist())

print(f"Selected {len(selected_indices)} test images ({N_PER_CLASS} per class).")

Selected 45 test images (15 per class).


In [12]:
pneumoxnet_results = []
start = time.time()

for i, idx in enumerate(selected_indices):
    image_tensor, true_label = test_dataset[idx]
    del_auc, ins_auc, pred_class = deletion_insertion_auc(pneumoxnet_model, pneumoxnet_gradcam, image_tensor, DEVICE)
    pneumoxnet_results.append({
        "model": "PneumoXNet",
        "index": idx,
        "true_label": CLASS_NAMES[true_label],
        "pred_label": CLASS_NAMES[pred_class],
        "deletion_auc": del_auc,
        "insertion_auc": ins_auc,
    })
    print(f"[PneumoXNet] {i+1}/{len(selected_indices)} | true={CLASS_NAMES[true_label]} pred={CLASS_NAMES[pred_class]} | del_auc={del_auc:.4f} ins_auc={ins_auc:.4f}")

print(f"\nPneumoXNet done in {(time.time()-start)/60:.1f} min")

[PneumoXNet] 1/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5753 ins_auc=0.7616
[PneumoXNet] 2/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5711 ins_auc=0.7173
[PneumoXNet] 3/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5381 ins_auc=0.7393
[PneumoXNet] 4/45 | true=BACTERIA pred=BACTERIA | del_auc=0.7193 ins_auc=0.8175
[PneumoXNet] 5/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5346 ins_auc=0.7990
[PneumoXNet] 6/45 | true=BACTERIA pred=VIRUS | del_auc=0.6879 ins_auc=0.6402
[PneumoXNet] 7/45 | true=BACTERIA pred=BACTERIA | del_auc=0.6776 ins_auc=0.7943
[PneumoXNet] 8/45 | true=BACTERIA pred=BACTERIA | del_auc=0.2478 ins_auc=0.7743
[PneumoXNet] 9/45 | true=BACTERIA pred=BACTERIA | del_auc=0.2626 ins_auc=0.5004
[PneumoXNet] 10/45 | true=BACTERIA pred=BACTERIA | del_auc=0.4519 ins_auc=0.7565
[PneumoXNet] 11/45 | true=BACTERIA pred=VIRUS | del_auc=0.7787 ins_auc=0.6878
[PneumoXNet] 12/45 | true=BACTERIA pred=VIRUS | del_auc=0.6685 ins_auc=0.5755
[PneumoXNet] 13/45 | true=BACTERIA pred=BACTER

In [13]:
baseline_results = []
start = time.time()

for i, idx in enumerate(selected_indices):
    image_tensor, true_label = test_dataset[idx]
    del_auc, ins_auc, pred_class = deletion_insertion_auc(baseline_model, baseline_gradcam, image_tensor, DEVICE)
    baseline_results.append({
        "model": "EfficientNet-B2",
        "index": idx,
        "true_label": CLASS_NAMES[true_label],
        "pred_label": CLASS_NAMES[pred_class],
        "deletion_auc": del_auc,
        "insertion_auc": ins_auc,
    })
    print(f"[Baseline] {i+1}/{len(selected_indices)} | true={CLASS_NAMES[true_label]} pred={CLASS_NAMES[pred_class]} | del_auc={del_auc:.4f} ins_auc={ins_auc:.4f}")

print(f"\nBaseline done in {(time.time()-start)/60:.1f} min")

[Baseline] 1/45 | true=BACTERIA pred=BACTERIA | del_auc=0.4886 ins_auc=0.7129
[Baseline] 2/45 | true=BACTERIA pred=BACTERIA | del_auc=0.4770 ins_auc=0.6238
[Baseline] 3/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5066 ins_auc=0.6768
[Baseline] 4/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5575 ins_auc=0.5827
[Baseline] 5/45 | true=BACTERIA pred=BACTERIA | del_auc=0.6573 ins_auc=0.6329
[Baseline] 6/45 | true=BACTERIA pred=BACTERIA | del_auc=0.3480 ins_auc=0.4156
[Baseline] 7/45 | true=BACTERIA pred=BACTERIA | del_auc=0.4356 ins_auc=0.6695
[Baseline] 8/45 | true=BACTERIA pred=BACTERIA | del_auc=0.3853 ins_auc=0.5439
[Baseline] 9/45 | true=BACTERIA pred=BACTERIA | del_auc=0.3308 ins_auc=0.4853
[Baseline] 10/45 | true=BACTERIA pred=BACTERIA | del_auc=0.5025 ins_auc=0.5384
[Baseline] 11/45 | true=BACTERIA pred=VIRUS | del_auc=0.6095 ins_auc=0.8404
[Baseline] 12/45 | true=BACTERIA pred=VIRUS | del_auc=0.3058 ins_auc=0.6292
[Baseline] 13/45 | true=BACTERIA pred=BACTERIA | del_auc=0.4521 ins

In [14]:
all_results_df = pd.DataFrame(pneumoxnet_results + baseline_results)
all_results_df.to_csv(XAI_METRICS_CSV, index=False)

summary = all_results_df.groupby("model")[["deletion_auc", "insertion_auc"]].agg(["mean", "std"])
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.round(4)

print("=" * 70)
print("Quantitative XAI Metrics — Deletion/Insertion AUC")
print("(Lower Deletion AUC = better | Higher Insertion AUC = better)")
print("=" * 70)
print(summary.to_string())
print("=" * 70)
print(f"Saved to: {XAI_METRICS_CSV}")

Quantitative XAI Metrics — Deletion/Insertion AUC
(Lower Deletion AUC = better | Higher Insertion AUC = better)
                 deletion_auc_mean  deletion_auc_std  insertion_auc_mean  insertion_auc_std
model                                                                                      
EfficientNet-B2             0.4641            0.1222              0.7031             0.1370
PneumoXNet                  0.5374            0.1717              0.7390             0.0873
Saved to: /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/results/xai_deletion_insertion_metrics.csv
